In [12]:
import json
import io
import shutil
import pickle
from pathlib import Path

from PIL import Image
import ipywidgets as widgets
from IPython.display import display

# --- 1. Configuration ---
COMPONENT = "clarifier"
ARCHITECTURE = "convnext_tiny"      # must match save_name used in save_result()
TEST_SET_ID = "Waterval"     # must match the TEST_SET_ID printed when wwtw_utils was imported

RESULTS_DIR = Path("../results")
RESULTS_FILE = RESULTS_DIR / f"results_{COMPONENT}_{ARCHITECTURE}_{TEST_SET_ID}.pkl"

# Written by prep.py: maps every crop filename back to the exact VIA2 json
# file / entry / region it came from, so a relabel here can be written back
# to the source annotation without reverse-parsing the crop filename.
CROP_MANIFEST_FILE = Path("../cnn_dataset/crop_manifest.json")

# Flags are keyed by component + test set (not architecture) — a ground
# truth mistake is a property of the image, not of which model happened to
# get it wrong, so flags accumulate into one file even if you review
# several architectures' results against the same held-out facility.
FLAGS_FILE = Path(f"./relabel_flags_{COMPONENT}_{TEST_SET_ID}.json")

THUMB_SIZE = (160, 160)
BATCH_SIZE = 50
GRID_COLS = 5

# The selectable relabel classes per component. Must match prep.py's
# valid/excluded class sets (AEROBIC_CLASSES/CLARIFIER_CLASSES minus their
# EXCLUDED_* sets), since a class that prep.py never saves to disk can't be
# picked here either.
CLASSES_BY_COMPONENT = {
    "aerobic_zone": ["Functional", "Suboptimal", "Dysfunctional"],
    "clarifier": ["Functional", "Dysfunctional", "Scum", "Empty"],
}
RELABEL_CLASSES = CLASSES_BY_COMPONENT[COMPONENT]

# Must match prep.py's AEROBIC_ATTR_KEY / CLARIFIER_ATTR_KEY.
ATTR_KEY_BY_COMPONENT = {
    "aerobic_zone": "aerobic zone",
    "clarifier": "clarifier",
}
ATTR_KEY = ATTR_KEY_BY_COMPONENT[COMPONENT]

# --- 2. Load results (no model, no inference) ---
if not RESULTS_FILE.exists():
    available = sorted(RESULTS_DIR.glob(f"results_{COMPONENT}_*.pkl"))
    raise FileNotFoundError(
        f"{RESULTS_FILE} not found. Available results files for component='{COMPONENT}':\n"
        + "\n".join(f"  {p.name}" for p in available)
    )

with open(RESULTS_FILE, "rb") as f:
    payload = pickle.load(f)

print(f"Loaded results: model={payload['model_name']} | component={payload['component']} | "
      f"test_set={payload.get('test_set_id', 'unknown')} | test_acc={payload['test_acc']:.3f}")

per_sample = payload["per_sample"]
misclassified = [item for item in per_sample if not item["correct"]]
print(f"Found {len(misclassified)} misclassified images (of {len(per_sample)} total test images).")

n_batches = max(1, -(-len(misclassified) // BATCH_SIZE))  # ceil div

# --- 3. Load crop manifest ---
# Keyed by crop filename -> {facility, unit_label, attr_key, json_path,
# img_key, region_index, label_at_export}. Only the sub-dict for this
# COMPONENT is relevant (aerobic_zone and clarifier crops can share a
# filename, since region indices restart at 0 per json file).
if CROP_MANIFEST_FILE.exists():
    with open(CROP_MANIFEST_FILE, "r", encoding="utf-8") as f:
        full_manifest = json.load(f)
    crop_manifest = full_manifest.get(COMPONENT, {})
    print(f"Loaded crop manifest: {len(crop_manifest)} {COMPONENT} crops tracked.")
else:
    crop_manifest = {}
    print(
        f"[warn] {CROP_MANIFEST_FILE} not found — relabel buttons will be disabled "
        f"for every image. Re-run the updated prep.py once to generate it."
    )

# --- 4. Load Existing Flags ---
# Tolerate a missing, empty, or corrupt flags file instead of crashing.
flags = {}
if FLAGS_FILE.exists():
    if FLAGS_FILE.stat().st_size == 0:
        print(f"[info] {FLAGS_FILE} exists but is empty — starting with a fresh flags dict.")
    else:
        try:
            with open(FLAGS_FILE, "r") as f:
                flags = json.load(f)
        except json.JSONDecodeError as e:
            print(f"[warn] {FLAGS_FILE} is not valid JSON ({e}) — starting with a fresh flags dict. "
                  f"The corrupt file was left untouched; delete it manually if you don't need it.")


def save_flags():
    with open(FLAGS_FILE, "w") as f:
        json.dump(flags, f, indent=2)


# --- 5. VIA2 working-copy helpers ---
# The first time any image from a given source json file gets relabeled, a
# "relabel" folder is created next to that facility's Images/Labels
# folders, holding a copy of the ORIGINAL json. Every subsequent relabel for
# that same json file edits that same copy in place — the original under
# Labels/ is never touched.

def get_relabel_dir(json_path: Path) -> Path:
    # json_path looks like .../wastewater/{Facility}/Labels/{name}.json
    # so its facility folder is two levels up.
    facility_root = json_path.parent.parent
    relabel_dir = facility_root / "relabel"
    relabel_dir.mkdir(parents=True, exist_ok=True)
    return relabel_dir


def get_working_copy_path(json_path: Path) -> Path:
    relabel_dir = get_relabel_dir(json_path)
    working_copy = relabel_dir / json_path.name
    if not working_copy.exists():
        shutil.copy2(json_path, working_copy)
        print(f"[info] created relabel working copy: {working_copy}")
    return working_copy


def apply_relabel(crop_info: dict, new_label: str) -> Path:
    """Writes new_label into the working copy's region_attributes for this
    crop's region, creating the working copy first if needed. Returns the
    working copy path that was edited."""
    json_path = Path(crop_info["json_path"])
    working_copy_path = get_working_copy_path(json_path)

    with open(working_copy_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    img_metadata = data.get("_via_img_metadata", data)

    entry = img_metadata[crop_info["img_key"]]
    region = entry["regions"][crop_info["region_index"]]
    region["region_attributes"][crop_info["attr_key"]] = new_label

    with open(working_copy_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    return working_copy_path


# --- 6. Widget State ---
state = {"batch_idx": 0}


def make_thumb_bytes(img_path):
    img = Image.open(img_path).convert("RGB")
    img.thumbnail(THUMB_SIZE)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def make_item_widget(item):
    path_str = item["path"]
    crop_filename = Path(path_str).name
    crop_info = crop_manifest.get(crop_filename)

    img_widget = widgets.Image(value=make_thumb_bytes(path_str), format="png",
                                width=THUMB_SIZE[0], height=THUMB_SIZE[1])

    label_html = widgets.HTML(
        f"<div style='font-size:11px'>"
        f"<b>True:</b> {item['true']}<br>"
        f"<b>Pred:</b> {item['pred']}<br>"
        f"<span style='color:gray'>{Path(path_str).name}</span>"
        f"</div>"
    )

    selection = {"current": flags.get(path_str, {}).get("relabel_to")}
    class_buttons = {}

    def refresh_button_styles():
        for cls_name, btn in class_buttons.items():
            btn.style.button_color = "#4caf50" if cls_name == selection["current"] else None
            btn.description = f"✓ {cls_name}" if cls_name == selection["current"] else cls_name

    def on_class_click(cls_name):
        def _handler(_):
            if crop_info is None:
                return
            if selection["current"] == cls_name:
                # Clicking the already-selected class again reverts back to
                # the label it had at export time.
                apply_relabel(crop_info, crop_info["label_at_export"])
                flags.pop(path_str, None)
                selection["current"] = None
            else:
                apply_relabel(crop_info, cls_name)
                flags[path_str] = {
                    "true": item["true"],
                    "pred": item["pred"],
                    "relabel_to": cls_name,
                    "source_model": payload["model_name"],
                    "test_set_id": payload.get("test_set_id"),
                    "json_path": crop_info["json_path"],
                    "img_key": crop_info["img_key"],
                    "region_index": crop_info["region_index"],
                }
                selection["current"] = cls_name
            save_flags()
            refresh_button_styles()
            update_status()
        return _handler

    for cls_name in RELABEL_CLASSES:
        btn = widgets.Button(
            description=cls_name,
            layout=widgets.Layout(width=f"{THUMB_SIZE[0] // 2 - 4}px"),
            disabled=(crop_info is None),
        )
        btn.on_click(on_class_click(cls_name))
        class_buttons[cls_name] = btn
    refresh_button_styles()

    button_grid = widgets.GridBox(
        list(class_buttons.values()),
        layout=widgets.Layout(grid_template_columns="repeat(2, auto)"),
    )

    children = [img_widget, label_html, button_grid]
    if crop_info is None:
        children.append(widgets.HTML(
            "<span style='color:#b00; font-size:10px;'>No manifest entry — "
            "re-run prep.py to enable relabeling for this image.</span>"
        ))

    return widgets.VBox(
        children,
        layout=widgets.Layout(border="1px solid #ddd", padding="4px", margin="2px", width=f"{THUMB_SIZE[0]+16}px")
    )


# --- 7. Rendering ---
output_area = widgets.Output()
status_label = widgets.Label()
batch_label = widgets.Label()


def update_status():
    status_label.value = f"Relabeled: {sum(1 for v in flags.values() if v.get('relabel_to'))} / {len(misclassified)}"


def render_batch(idx):
    idx = max(0, min(idx, n_batches - 1))
    state["batch_idx"] = idx

    start = idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(misclassified))
    batch_items = misclassified[start:end]

    batch_label.value = f"Batch {idx + 1}/{n_batches}  (items {start + 1}-{end} of {len(misclassified)})"

    item_widgets = [make_item_widget(item) for item in batch_items]
    grid = widgets.GridBox(
        item_widgets,
        layout=widgets.Layout(grid_template_columns=f"repeat({GRID_COLS}, auto)")
    )

    with output_area:
        output_area.clear_output(wait=True)
        display(grid)

    update_status()


# --- 8. Controls ---
prev_btn = widgets.Button(description="⬅ Previous", button_style="")
next_btn = widgets.Button(description="Next ➡", button_style="")
export_btn = widgets.Button(description="Export relabeled list (CSV)", button_style="info")
export_output = widgets.Output()


def on_prev(_):
    render_batch(state["batch_idx"] - 1)


def on_next(_):
    render_batch(state["batch_idx"] + 1)


def on_export(_):
    import csv
    out_path = Path(f"./relabeled_{COMPONENT}_{TEST_SET_ID}.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "path", "true_label", "predicted_label", "relabel_to",
            "source_model", "test_set_id", "json_path", "img_key", "region_index",
        ])
        for path_str, info in flags.items():
            writer.writerow([
                path_str, info["true"], info["pred"], info.get("relabel_to", ""),
                info.get("source_model", ""), info.get("test_set_id", ""),
                info.get("json_path", ""), info.get("img_key", ""), info.get("region_index", ""),
            ])
    with export_output:
        export_output.clear_output(wait=True)
        print(f"Exported {len(flags)} relabeled items to {out_path}")


prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
export_btn.on_click(on_export)

controls = widgets.HBox([prev_btn, batch_label, next_btn])
top_bar = widgets.HBox([status_label, export_btn])

# --- 9. Display ---
display(widgets.VBox([top_bar, controls, output_area, export_output]))
render_batch(0)


Loaded results: model=ConvNeXt-Tiny | component=clarifier | test_set=Waterval | test_acc=0.736
Found 75 misclassified images (of 284 total test images).
Loaded crop manifest: 2507 clarifier crops tracked.
